# Feature Decisions

Objective:
- Finalize feature inclusion/exclusion decisions
- Define transformations (without implementing them)
- Identify encoding and scaling strategies
- Document rationale for each decision

Rules:
- No feature construction code
- No model training
- Decisions must be defensible and auditable


In [2]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)

In [3]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_path = os.path.join(project_root, 'data', 'interim', 'cleaned_data.csv')

crime_data = pd.read_csv(data_path)
crime_data.shape

(10000, 10)

In [4]:
# Define target and exclude columns
TARGET_COL = "crime_type"

EXCLUDE_COLS = [
    "crime_id"
]

df = crime_data.drop(columns=EXCLUDE_COLS)
df.columns


Index(['crime_type', 'latitude', 'longitude', 'hour', 'day_of_week',
       'victim_age', 'suspect_age', 'weapon_used', 'arrest_made'],
      dtype='object')

## Feature Inventory

### Numeric
- hour
- latitude
- longitude
- victim_age
- suspect_age

### Categorical
- weapon_used
- day_of_week
- neighborhood

### Target
- crime_type


## Numeric Feature Decisions

| Feature        | Keep | Transformation | Rationale |
|---------------|------|----------------|-----------|
| hour          | Yes  | Cyclical sin/cos | Periodic behavior |
| latitude      | Yes  | Spatial binning | Raw values weak |
| longitude     | Yes  | Spatial binning | Raw values weak |
| victim_age    | Yes  | Binning         | Reduce noise/bias |
| suspect_age   | Yes  | Binning         | Reduce noise/bias |


## Categorical Feature Decisions

| Feature       | Keep | Encoding | Notes |
|--------------|------|----------|------|
| weapon_used  | Yes  | One-hot (group rare) | 'unknown' preserved |
| day_of_week  | Yes  | One-hot | Low cardinality |
| neighborhood | Yes  | Target/frequency | High cardinality |


## Excluded Features

| Feature   | Reason |
|----------|--------|
| crime_id | Identifier |


## Leakage Safety Confirmation

- No feature derived from target
- No post-outcome attributes included
- Temporal features validated
- All transformations are target-agnostic


## Bias Considerations

- Age features will be binned to reduce individual-level impact
- Geographic features binned to reduce proxy bias
- Sensitive attributes retained with caution and monitoring


## Feature Schema Contract (Processed Data)

| Feature Name        | Type      |
|--------------------|-----------|
| hour_sin           | float |
| hour_cos           | float |
| lat_bin            | category |
| lon_bin            | category |
| victim_age_bin     | category |
| suspect_age_bin    | category |
| weapon_used_*      | binary |
| day_of_week_*      | binary |
| neighborhood_enc   | float |
| crime_type         | category |


## Pipeline Handoff

These decisions will be implemented in:
- features/build_features.py
- pipelines/training_pipeline.py
- validation/model_validation.py

Any deviation requires re-approval.


## Final Decisions

- All features retained with transformations
- No features dropped beyond identifiers
- Encoding strategies fixed
- Ready for feature engineering implementation
